notebook_ft_finetuning.ipynb

Fine-tuning Preparation + Submission
Input:  
         
         cat1_usage_product_real.jsonl
         cat2_diagnosis_real.jsonl
         cat3_aftersales_logistics_real.jsonl
         cat4_safety_sensitive_real.jsonl

 Output: agromind_training_v2.jsonl   (cleaned, system-prompted)

         fine_tuning_job.json         (job ID + status)

In [ ]:
# =============================================================================
# notebook_ft_gemini.ipynb
# Gemini Fine-tuning via Vertex AI
# Input:  agromind_training_v2.jsonl  (from notebook_ft_finetuning)
# Output: agromind_training_gemini.jsonl  (converted format)
#         gemini_tuning_job.json          (job ID + tuned model name)
# =============================================================================
# Requirements:
#   - Google Cloud project with billing enabled
#   - Vertex AI API enabled
#   - Cloud Storage bucket
#   - Gemini API key (for inference testing after tuning)
# =============================================================================

In [ ]:
!pip install google-cloud-aiplatform google-cloud-storage -q

In [ ]:
# ── CELL 1 — Imports + API key ────────────────────────────────────────────────
import os, json
from google.colab import files, auth, userdata
from google.cloud import storage
import vertexai

auth.authenticate_user()

GCP_PROJECT  = userdata.get('GCP_PROJECT')
GCS_BUCKET   = userdata.get('GCS_BUCKET')
GOOGLE_KEY   = userdata.get('GOOGLE_API_KEY')
GCP_LOCATION = 'us-central1'
GCS_PREFIX   = 'agromind/training'

os.environ['GOOGLE_API_KEY'] = GOOGLE_KEY

storage_client = storage.Client(project=GCP_PROJECT)
vertexai.init(project=GCP_PROJECT, location=GCP_LOCATION)

print(f'Project:  {GCP_PROJECT}')
print(f'Bucket:   gs://{GCS_BUCKET}')
print(f'Location: {GCP_LOCATION}')
print('Auth + config ready ✓')

Project:  gen-lang-client-0396125347
Bucket:   gs://agromind-ohoud-2026
Location: us-central1
Auth + config ready ✓


In [ ]:
# ── CELL 2 — Upload training file ────────────────────────────────────────────
print('Upload agromind_training_v2.jsonl')
uploaded = files.upload()

with open('agromind_training_v2.jsonl', encoding='utf-8') as f:
    openai_examples = [json.loads(l) for l in f if l.strip()]

print(f'Loaded {len(openai_examples)} examples')

Upload agromind_training_v2.jsonl


Saving agromind_training_v2.jsonl to agromind_training_v2.jsonl
Loaded 234 examples


In [ ]:
# ── CELL 3 — Convert OpenAI → Gemini format + fix last-turn issue ─────────────
# OpenAI: messages:[{role, content}]  system as first message
# Gemini: systemInstruction:{parts:[{text}]} + contents:[{role:user/model, parts:[{text}]}]
# Fix:    every example must end with a "model" turn

def convert_to_gemini(ex):
    system_text = None
    contents    = []
    for msg in ex['messages']:
        if msg['role'] == 'system':
            system_text = msg['content']
        elif msg['role'] == 'user':
            contents.append({'role': 'user',  'parts': [{'text': msg['content']}]})
        elif msg['role'] == 'assistant':
            contents.append({'role': 'model', 'parts': [{'text': msg['content']}]})
    result = {'contents': contents}
    if system_text:
        result['systemInstruction'] = {'parts': [{'text': system_text}]}
    return result

converted = [convert_to_gemini(ex) for ex in openai_examples]

# Fix: trim any example that doesn't end with a model turn
fixed   = []
removed = []
for i, ex in enumerate(converted):
    contents = ex['contents']
    if not contents:
        removed.append(i)
        continue
    if contents[-1]['role'] == 'model':
        fixed.append(ex)
    else:
        # Trim to last model turn
        model_indices = [j for j, c in enumerate(contents) if c['role'] == 'model']
        if model_indices:
            ex_copy = dict(ex)
            ex_copy['contents'] = contents[:model_indices[-1] + 1]
            fixed.append(ex_copy)
        else:
            removed.append(i)

print(f'Converted:       {len(converted)}')
print(f'Valid after fix: {len(fixed)}')
print(f'Removed:         {len(removed)}')

# Spot check
sample = fixed[0]
print(f'\nSample — last turn role: {sample["contents"][-1]["role"]}  ← must be "model"')
print(f'Sample — first user: {sample["contents"][0]["parts"][0]["text"][:60]}')

# Save
FIXED_FILE = 'agromind_training_gemini_v2.jsonl'
with open(FIXED_FILE, 'w', encoding='utf-8') as f:
    for ex in fixed:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print(f'\nSaved → {FIXED_FILE}')
files.download(FIXED_FILE)



Converted:       234
Valid after fix: 234
Removed:         0

Sample — last turn role: model  ← must be "model"
Sample — first user: 4克兑水一斤什么意思？

Saved → agromind_training_gemini_v2.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── CELL 4 — Upload fixed file to Cloud Storage ───────────────────────────────
try:
    bucket = storage_client.get_bucket(GCS_BUCKET)
    print(f'Using existing bucket: gs://{GCS_BUCKET}')
except Exception:
    bucket = storage_client.create_bucket(GCS_BUCKET, location=GCP_LOCATION)
    print(f'Created bucket: gs://{GCS_BUCKET}')

blob_name = f'{GCS_PREFIX}/agromind_training_gemini_v2.jsonl'
blob      = bucket.blob(blob_name)
blob.upload_from_filename(FIXED_FILE)
GCS_URI   = f'gs://{GCS_BUCKET}/{blob_name}'
print(f'Uploaded → {GCS_URI}')

Using existing bucket: gs://agromind-ohoud-2026
Uploaded → gs://agromind-ohoud-2026/agromind/training/agromind_training_gemini_v2.jsonl


In [ ]:
# ── CELL 5 — Submit fine-tuning job ──────────────────────────────────────────
from vertexai.preview.tuning import sft

print('Submitting fine-tuning job...')
tuning_job = sft.train(
    source_model             = 'gemini-2.5-flash',
    train_dataset            = GCS_URI,
    epochs                   = 3,
    tuned_model_display_name = 'agromind-support-agent-v2',
)

print(f'Job submitted ✓')
print(f'Job name: {tuning_job.resource_name}')
print(f'Status:   {tuning_job.state}')

job_info = {
    'job_name':         tuning_job.resource_name,
    'base_model':       'gemini-2.5-flash',
    'training_data':    GCS_URI,
    'examples':         len(fixed),
    'status':           str(tuning_job.state),
    'tuned_model_name': None,
}
with open('gemini_tuning_job_v2.json', 'w') as f:
    json.dump(job_info, f, indent=2)
files.download('gemini_tuning_job_v2.json')

print("""
=== JOB SUBMITTED ✓ ===
Takes: 1-3 hours
Run Cell 6 to check status.
""")

INFO:vertexai.tuning._tuning:Creating SupervisedTuningJob


Submitting fine-tuning job...


INFO:vertexai.tuning._tuning:SupervisedTuningJob created. Resource name: projects/548742106129/locations/us-central1/tuningJobs/2210300929138229248
INFO:vertexai.tuning._tuning:To use this SupervisedTuningJob in another session:
INFO:vertexai.tuning._tuning:tuning_job = sft.SupervisedTuningJob('projects/548742106129/locations/us-central1/tuningJobs/2210300929138229248')
INFO:vertexai.tuning._tuning:View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/us-central1/tuning/tuningJob/2210300929138229248?project=548742106129


Job submitted ✓
Job name: projects/548742106129/locations/us-central1/tuningJobs/2210300929138229248
Status:   2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== JOB SUBMITTED ✓ ===
Takes: 1-3 hours
Run Cell 6 to check status.



In [32]:
import json
with open('gemini_tuning_job_v2.json') as f:
    info = json.load(f)
print(json.dumps(info, indent=2))

{
  "job_name": "projects/548742106129/locations/us-central1/tuningJobs/2210300929138229248",
  "base_model": "gemini-2.5-flash",
  "training_data": "gs://agromind-ohoud-2026/agromind/training/agromind_training_gemini_v2.jsonl",
  "examples": 234,
  "status": "2",
  "tuned_model_name": null
}


In [34]:
# ── CELL 6 — Check status + get tuned model name ─────────────────────────────
import json
import google.generativeai as genai
from google.colab import files
from vertexai.preview.tuning import sft
import os

# Load job info
with open('gemini_tuning_job_v2.json') as f:
    job_info = json.load(f)

print(f'Job: {job_info["job_name"]}')

# Refresh status from Vertex AI
tuning_job = sft.SupervisedTuningJob(job_info['job_name'])
tuning_job.refresh()
print(f'Status: {tuning_job.state}')

if hasattr(tuning_job, 'tuned_model_name') and tuning_job.tuned_model_name:
    tuned_model = tuning_job.tuned_model_name
    print(f'\nTuned model: {tuned_model}')
    print('Copy this name — you need it for notebook_08_agent.ipynb')

    # Save updated job info
    job_info['tuned_model_name'] = tuned_model
    job_info['status']           = str(tuning_job.state)
    with open('gemini_tuning_job_v2.json', 'w') as f:
        json.dump(job_info, f, indent=2)
    files.download('gemini_tuning_job_v2.json')

    # Quick test
    genai.configure(api_key=os.environ.get('GOOGLE_API_KEY', ''))
    model    = genai.GenerativeModel(tuned_model)
    response = model.generate_content('打完药几天可以吃菜？')
    print(f'\nTest query: 打完药几天可以吃菜？')
    print(f'Response:   {response.text}')

else:
    print('Still running — check again in a few minutes.')

Job: projects/548742106129/locations/us-central1/tuningJobs/2210300929138229248


/usr/local/lib/python3.12/dist-packages/vertexai/tuning/_tuning.py:110: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Status: 4

Tuned model: projects/548742106129/locations/us-central1/models/6556995316602634240@1
Copy this name — you need it for notebook_08_agent.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ValueError: Invalid request.
Some of the fields of the request message are either not initialized or initialized with an invalid value.
Please make sure your request matches at least one accepted HTTP binding.
To match a binding the request message must have all the required fields initialized with values matching their patterns as listed below:
	URI: "/v1beta/{model=models/*}:generateContent"
	Required request fields:
		field: "model", pattern: "models/*"

	URI: "/v1beta/{model=tunedModels/*}:generateContent"
	Required request fields:
		field: "model", pattern: "tunedModels/*"

In [41]:
# Test tuned model via Vertex AI
import vertexai
from google.cloud import aiplatform

vertexai.init(project=GCP_PROJECT, location=GCP_LOCATION)

# Use the new google.genai SDK instead of deprecated vertexai
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)

response = client.models.generate_content(
    model=tuned_model,
    contents='打完药几天可以吃菜？'
)
print(f'Test query: 打完药几天可以吃菜？')
print(f'Response:   {response.text}')

ClientError: 404 Not Found. {'message': '', 'status': 'Not Found'}